# u3lib — Python tutorial

Bindings for the header-only C++ **u3lib** (SU(3)/U(3) coupling & recoupling
coefficients). The Python names mirror the C++ API 1:1. Build the module first
with `./build.sh`; the bindings live in [pyu3lib.cpp](pyu3lib.cpp) and build
details in [BUILD_NOTES.md](BUILD_NOTES.md).

This notebook exercises the **whole current interface** so you can confirm it
before we move on to u4lib. Recognizable spot-checks are called out in the text.

| Python entry point | Purpose |
|---|---|
| `su3.irrep(λ, μ)`, `irrep(λ, μ)`, `irrep(n1, n2, n3)` | SU(3) / U(3) irreps |
| `gelfand(n13,n23,n33,n12,n22,n11)` | Gelfand–Tsetlin state + generator MEs |
| `Rho(a, b, c)` | Outer coupling multiplicity |
| `wigU3U2db().get(...)` | U(3) ⊃ U(2) reduced coefficients (numpy row) |
| `wigU3SO3db().get(...)` / `.get_U3SO3(...)` | U(3) ⊃ SO(3), flat or structured |
| `u3u6db()/u3z6db()/u3u9db().get_U3*(...)` | 6-/9-(λμ) recoupling (structured) |
| `initialize()`, `clear()`, `finalize()` | Session + cache management |

> **Asserts are compiled in** (no `-DNDEBUG`). Indexing an *empty / forbidden*
> coupling, or passing invalid labels, will `abort()` the kernel rather than
> raise a Python exception. Guard with `.empty()` or check `Rho(...)` first.

## 1. Import the extension

Finds `u3lib.so` whether Jupyter is launched from `QCX`, `u3lib`, or
`u3lib/python`, and prints the exact path so no installed package shadows it.

In [2]:
from pathlib import Path
import sys
import numpy as np
from IPython.display import Markdown, display

candidates = []
for base in (Path.cwd(), *Path.cwd().parents):
    candidates.extend((base, base / "python", base / "u3lib" / "python"))
python_dir = next((p.resolve() for p in candidates
                   if (p / "pyu3lib.cpp").is_file()), None)
if python_dir is None:
    raise FileNotFoundError("Launch Jupyter from QCX, u3lib, or u3lib/python.")

sys.path.insert(0, str(python_dir))
import u3lib as u3
if Path(u3.__file__).resolve().parent != python_dir:
    raise RuntimeError("Another u3lib module is loaded. Restart the kernel.")

np.set_printoptions(precision=10, suppress=True)
print("Python:", sys.version.split()[0])
print("Extension:", u3.__file__)

def table(headers, rows):
    lines = ["| " + " | ".join(map(str, headers)) + " |",
             "| " + " | ".join(["---"] * len(headers)) + " |"]
    lines += ["| " + " | ".join(map(str, r)) + " |" for r in rows]
    display(Markdown("\n".join(lines)))

Python: 3.14.6
Extension: /Users/phongdang/Desktop/QCX/u3lib/python/u3lib.so


## 2. Start a session

`initialize()` sets up the qcx-utils foundation (Wigner tables + SO(3) caches)
**and** the U(3) coefficient databases. Always pair with `finalize()`.

In [3]:
u3.initialize()
print("u3lib session initialized.")

u3lib session initialized.


## 3. Irreps

- `su3.irrep(λ, μ)` — an SU(3) irrep; `str()` → `(λ,μ)`.
- `irrep(λ, μ)` — a U(3) irrep from SU(3) labels (n33 = 0); `str()` → `[n13,n23,n33]`.
- `irrep(n1, n2, n3)` — a U(3) irrep from the three row labels (n1 ≥ n2 ≥ n3).

`u3.irrep` derives from `su3.irrep`. Use `.lam()` / `.lambda_()` (`lambda` is a
Python keyword). Dimension: d(λ,μ) = ½(λ+1)(μ+1)(λ+μ+2).

In [4]:
labels = [(0, 0), (1, 0), (0, 1), (2, 0), (1, 1), (4, 2)]
rows = []
for lam, mu in labels:
    ir = u3.irrep(lam, mu)
    expected = (lam + 1) * (mu + 1) * (lam + mu + 2) // 2
    assert ir.dim() == expected
    rows.append((f"({lam},{mu})", str(ir), (ir.n13(), ir.n23(), ir.n33()), ir.dim()))
table(["SU(3) (λ,μ)", "U(3) str", "(n13,n23,n33)", "dim"], rows)

# Two ways to build the same U(3) irrep:
print("irrep(4,2)   ->", str(u3.irrep(4, 2)))          # [6,2,0]
print("irrep(6,4,2) ->", str(u3.irrep(6, 4, 2)),
      "  (lam,mu) =", u3.irrep(6, 4, 2).lam(), u3.irrep(6, 4, 2).mu())

# su3.irrep: Casimirs and conjugation
s = u3.su3.irrep(4, 2)
print("su3.irrep(4,2):", str(s), "| dim", s.dim(),
      "| C2", round(s.C2(), 6), "| C3", round(s.C3(), 6),
      "| conj", str(s.conjugate()))
assert u3.irrep(0, 0).zero()

| SU(3) (λ,μ) | U(3) str | (n13,n23,n33) | dim |
| --- | --- | --- | --- |
| (0,0) | [0,0,0] | (0, 0, 0) | 1 |
| (1,0) | [1,0,0] | (1, 0, 0) | 3 |
| (0,1) | [1,1,0] | (1, 1, 0) | 3 |
| (2,0) | [2,0,0] | (2, 0, 0) | 6 |
| (1,1) | [2,1,0] | (2, 1, 0) | 8 |
| (4,2) | [6,2,0] | (6, 2, 0) | 60 |

irrep(4,2)   -> [6,2,0]
irrep(6,4,2) -> [6,4,2]   (lam,mu) = 2 2
su3.irrep(4,2): (4,2) | dim 60 | C2 30.666667 | C3 31.777778 | conj (2,4)


## 4. Gelfand–Tsetlin states

`gelfand(n13, n23, n33, n12, n22, n11)` — carries the betweenness check, the
SU(2) subgroup labels (ε, 2Λ, 2M_Λ), and the U(3) **generator matrix elements**
`e11`…`e31_22` used when building reduced coefficients.

In [5]:
g = u3.gelfand(6, 2, 0, 4, 2, 3)
print("state     :", str(g), "| repr", repr(g))
print("valid?    :", g.check())
print("lam, mu   :", g.lam(), g.mu())
print("epsilon   :", g.epsilon(), "| 2*Lambda", g.Lambdax2(), "| 2*M_Lambda", g.MLambdax2())
print("e11,e22,e33:", g.e11(), g.e22(), g.e33())
print("e12, e21  :", g.e12(), g.e21())
print("conjugate :", str(g.conjugate()), "| phase", g.conjugate_phase())

state     : 6,2,0;4,2;3 | repr gelfand(6,2,0;4,2;3)
valid?    : True
lam, mu   : 4 2
epsilon   : -2 | 2*Lambda 2 | 2*M_Lambda 0
e11,e22,e33: 3.0 3.0 2.0
e12, e21  : 1.4142135623730951 1.4142135623730951
conjugate : 6,4,0;4,2;3 | phase -1.0


## 5. Outer multiplicity `Rho`

`Rho(a, b, c)` = number of copies of `c` in `a × b`. Two overloads: three
`irrep` objects, or six SU(3) labels.

Spot-checks: in 8⊗8 the octet appears **twice**; the total dimension is 64.
Triality forbids `(4,2)×(4,2)→(4,2)` → **0**.

In [6]:
adjoint = u3.irrep(1, 1)
channels = [(0, 0), (1, 1), (3, 0), (0, 3), (2, 2), (3, 3)]
rows = []
for lam, mu in channels:
    c = u3.irrep(lam, mu)
    rmax = u3.Rho(adjoint, adjoint, c)
    rows.append((f"({lam},{mu})", c.dim(), rmax, c.dim() * rmax))
table(["output (λ,μ)", "dim", "multiplicity", "dim × mult"], rows)
assert u3.Rho(adjoint, adjoint, adjoint) == 2
assert sum(r[-1] for r in rows) == adjoint.dim() ** 2 == 64
print("8 x 8: octet multiplicity =", u3.Rho(1,1, 1,1, 1,1), "; dim check 64 = 8*8 OK")

print("Rho (2,0)x(2,0)->(4,0):", u3.Rho(2,0, 2,0, 4,0))
print("Rho (4,2)x(4,2)->(4,2):", u3.Rho(4,2, 4,2, 4,2), "(triality-forbidden)")

| output (λ,μ) | dim | multiplicity | dim × mult |
| --- | --- | --- | --- |
| (0,0) | 1 | 1 | 1 |
| (1,1) | 8 | 2 | 16 |
| (3,0) | 10 | 1 | 10 |
| (0,3) | 10 | 1 | 10 |
| (2,2) | 27 | 1 | 27 |
| (3,3) | 64 | 0 | 0 |

8 x 8: octet multiplicity = 2 ; dim check 64 = 8*8 OK
Rho (2,0)x(2,0)->(4,0): 1
Rho (4,2)x(4,2)->(4,2): 0 (triality-forbidden)


## 6. U(3) ⊃ U(2) coefficients — `wigU3U2db`

Per irrep a/b/c, `get(...)` takes `(n13, n23, n33, n12, n22)` and returns the
row over ρ as a 1-D numpy array (index 0 ↔ ρ=1). Interlacing:
n13 ≥ n12 ≥ n23 ≥ n22 ≥ n33.

The example uses the same labels as the C++ U(3) coupling test — each irrep is
`[4,2,0]`, i.e. (λ,μ)=(2,2), with ρ_max = 3.

In [7]:
state_a = (4, 2, 0, 4, 2)
state_b = (4, 2, 0, 4, 0)
state_c = (4, 2, 0, 4, 2)
for n13, n23, n33, n12, n22 in (state_a, state_b, state_c):
    assert n13 >= n12 >= n23 >= n22 >= n33

u3u2_db = u3.wigU3U2db()
u3u2_row = np.asarray(u3u2_db.get(*state_a, *state_b, *state_c))
assert u3u2_row.shape == (u3.Rho(2,2, 2,2, 2,2),)
table(["ρ", "reduced coefficient"],
      [(rho, f"{v:+.10f}") for rho, v in enumerate(u3u2_row, start=1)])

| ρ | reduced coefficient |
| --- | --- |
| 1 | +0.7745966692 |
| 2 | +0.1463850109 |
| 3 | +0.0243975018 |

## 7. U(3) ⊃ SO(3) coefficients — `wigU3SO3db`

Two readouts:
- `get(lam,mu,L, ...)` → flat numpy array.
- `get_U3SO3(...)` → a **U3SO3** object indexed as `obj(kappaA, kappaB, kappaC, rho)`,
  with shape getters `kappaAmax()/kappaBmax()/kappaCmax()/rhomax()` and `.data()`.

Spot-check: `(2,0)L2 × (2,0)L2 → (4,0)L2` gives `−√(2/9) ≈ −0.4714`.

In [8]:
u3so3_db = u3.wigU3SO3db()
u3so3 = u3so3_db.get_U3SO3(2,0,2,  2,0,2,  4,0,2)
print("empty? ", u3so3.empty(),
      "| shape (kA,kB,kC,rho) =", u3so3.kappaAmax(), u3so3.kappaBmax(), u3so3.kappaCmax(), u3so3.rhomax())
print(".data() =", u3so3.data())
print("obj(0,0,0,0) =", u3so3(0, 0, 0, 0), "  check -sqrt(2/9) =", -np.sqrt(2/9))
np.testing.assert_allclose(u3so3(0,0,0,0), -np.sqrt(2/9), atol=1e-10)

# A forbidden channel returns an empty object (guard before indexing):
forb = u3so3_db.get_U3SO3(2,0,0, 2,0,0, 4,0,2)
print("forbidden channel empty? ", forb.empty())

empty?  False | shape (kA,kB,kC,rho) = 1 1 1 1
.data() = [-0.4714045208]
obj(0,0,0,0) = -0.4714045207910319   check -sqrt(2/9) = -0.4714045207910317
forbidden channel empty?  True


## 8. A normalization check you can reproduce

For `(2,0)⊗(2,0)→(4,0)` every channel is multiplicity-free. For each output
`L_C`, the sum of squared reduced coefficients over the input pairs `(L_A,L_B)`
should be **1**.

In [10]:
from itertools import product
assert u3.Rho(2,0, 2,0, 4,0) == 1
coeff_rows, norms = [], []
for lc in (0, 2, 4):
    ns = 0.0
    for la, lb in product((0, 2), repeat=2):
        v = np.asarray(u3so3_db.get(2,0,la, 2,0,lb, 4,0,lc))
        if v.size == 0:
            continue
        coeff_rows.append((la, lb, lc, f"{float(v[0]):+.10f}"))
        ns += float(v[0]) ** 2
    norms.append((lc, ns))
table(["L_A", "L_B", "L_C", "reduced coefficient"], coeff_rows)
table(["L_C", "Σ coeff²"], [(lc, f"{n:.12f}") for lc, n in norms])
np.testing.assert_allclose([n for _, n in norms], np.ones(3), atol=1e-10)
print("All three normalization checks passed.")

| L_A | L_B | L_C | reduced coefficient |
| --- | --- | --- | --- |
| 0 | 0 | 0 | +0.7453559925 |
| 2 | 2 | 0 | +0.6666666667 |
| 0 | 2 | 2 | +0.6236095645 |
| 2 | 0 | 2 | +0.6236095645 |
| 2 | 2 | 2 | -0.4714045208 |
| 2 | 2 | 4 | +1.0000000000 |

| L_C | Σ coeff² |
| --- | --- |
| 0 | 1.000000000000 |
| 2 | 1.000000000000 |
| 4 | 1.000000000000 |

All three normalization checks passed.


## 9. Recoupling — `u3u6db` / `u3z6db` / `u3u9db`

The 6-(λμ) U- and Z-coefficients and the 9-(λμ) coefficients. Each `get_U3*`
takes the relevant coupling **multiplicities** (from `Rho`), mirroring the C++
signature, and returns a structured **U3U6 / U3Z6 / U3U9** object with
multiplicity-indexed `__call__` and `.data()`.

The examples use a scalar leg so every multiplicity is 1.

In [12]:
A = u3.irrep(2,0); B = u3.irrep(0,0); AB = u3.irrep(2,0)
C = u3.irrep(2,0); ABC = u3.irrep(4,0); BC = u3.irrep(2,0)
rAB, rAB_C = u3.Rho(A,B,AB), u3.Rho(AB,C,ABC)
rBC, rA_BC = u3.Rho(B,C,BC), u3.Rho(A,BC,ABC)
u3u6 = u3.u3u6db().get_U3U6(A, B, ABC, C, AB, rAB, rAB_C, BC, rBC, rA_BC)
print("U3U6 size", u3u6.size(), "| (0,0,0,0) =", u3u6(0,0,0,0))
print(u3u6)

# Z-coefficient: recouple (A x B), (AB x C) vs (A x C), (AC x B). Scalar B.
AC = u3.irrep(4,0)
rAC, rAC_B = u3.Rho(A,C,AC), u3.Rho(AC,B,ABC)
u3z6 = u3.u3z6db().get_U3Z6(B, A, ABC, C, AB, rAB, rAB_C, AC, rAC, rAC_B)
print("U3Z6 size", u3z6.size(), "| .data() =", u3z6.data())

U3U6 size 1 | (0,0,0,0) = 1.0
rhoAB	rhoAB_C	rhoBC	rhoA_BC
1	1	1	1	 -> 1

U3Z6 size 1 | .data() = [1.]


In [14]:
# 9-(λμ): four legs A,B,C,D; scalar B,C keep every multiplicity 1.
A = u3.irrep(2,0); B = u3.irrep(0,0); C = u3.irrep(0,0); D = u3.irrep(2,0)
AB = u3.irrep(2,0); CD = u3.irrep(2,0); AC = u3.irrep(2,0); BD = u3.irrep(2,0)
ABCD = u3.irrep(4,0)
rAB, rCD = u3.Rho(A,B,AB), u3.Rho(C,D,CD)
rAC, rBD = u3.Rho(A,C,AC), u3.Rho(B,D,BD)
rABCD, rACBD = u3.Rho(AB,CD,ABCD), u3.Rho(AC,BD,ABCD)
u3u9 = u3.u3u9db().get_U3U9(A,B,C,D, AB,CD, AC,BD, ABCD, rAC,rBD,rABCD, rAB,rCD,rACBD)
print("U3U9 size", u3u9.size(), "| (0,0,0,0,0,0) =", u3u9(0,0,0,0,0,0), "| .data()", u3u9.data())

U3U9 size 1 | (0,0,0,0,0,0) = 1.0 | .data() [1.]


## 10. Cache behavior

The databases cache computed coefficients. `clear()` discards cached values
while keeping the session initialized; the next request recomputes them.

In [16]:
ref = np.array(u3u2_db.get(*state_a, *state_b, *state_c), copy=True)
u3.clear()
again = np.asarray(u3u2_db.get(*state_a, *state_b, *state_c))
np.testing.assert_allclose(again, ref, atol=1e-10)
print("Repeated and recomputed results agree.")

Repeated and recomputed results agree.


## 11. Finish

Finalize after all calculations. Don't use the database objects afterward; to
run more coefficient cells, rerun the initialization cell (or restart & run all).

In [17]:
u3.finalize()
print("Session finalized. All tutorial checks passed.")

Session finalized. All tutorial checks passed.
